# Análisis integrado: MySQL y MongoDB

In [2]:
!pip install pymysql pymongo

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 984.5 kB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 1.5 MB/s  0:00:00

   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   -------------------------- ------------- 2/3 [pymongo]
   -------------------------- ------------- 2/3 [pymongo]
   -------------------------- ------------- 2/3 [pymongo]
   -------------------------- ------------- 2/3 [pymongo]
   -------------------------- ------------

In [ ]:
import os
import pandas as pd
import pymysql
from pymongo import MongoClient

mysql = pymysql.connect(
    host=os.getenv("MYSQL_HOST", "mysql"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    user=os.getenv("MYSQL_USER", "alumno"),
    password=os.getenv("MYSQL_PASSWORD", "alumno123"),
    database=os.getenv("MYSQL_DATABASE", "finanzas"),
)

mongo = MongoClient(os.getenv("MONGO_URI"))
coleccion = mongo[os.getenv("MONGO_DATABASE", "finanzas_nosql")][os.getenv("MONGO_COLLECTION", "movimientos")]
print("Conexiones creadas")

OperationalError: (2003, "Can't connect to MySQL server on 'mysql' ([Errno 11001] getaddrinfo failed)")

In [ ]:
consulta = """
SELECT DATE_FORMAT(fecha, '%Y-%m') AS anio_mes,
       SUM(CASE WHEN tipo='Ingreso' THEN importe ELSE 0 END) AS ingresos,
       SUM(CASE WHEN tipo='Gasto' THEN importe ELSE 0 END) AS gastos,
       SUM(CASE WHEN tipo='Ingreso' THEN importe ELSE -importe END) AS balance,
       COUNT(*) AS movimientos
FROM movimientos
GROUP BY DATE_FORMAT(fecha, '%Y-%m')
ORDER BY anio_mes
"""
balance = pd.read_sql(consulta, mysql)
balance

In [ ]:
documentos = list(coleccion.find({}, {"_id":0}))
df_mongo = pd.json_normalize(documentos)
df_mongo.head()

In [ ]:
ax = balance.set_index("anio_mes")[["ingresos", "gastos"]].plot(marker="o", figsize=(10,5), title="Ingresos y gastos mensuales")
ax.set_xlabel("Mes")
ax.set_ylabel("Importe (€)")